# Website Fingerprinting: Phase 2 - Baseline SVM Training

In this notebook, we transition from raw sequential packet directions to partitioning our dataset and training our baseline model. Establishing a baseline provides a critical benchmark to measure the performance of the 1D-Convolutional Neural Network (1D-CNN) architecture later.

Using scikit-learn, we divide the feature matrix into 60% Training / 20% Validation / 20% Testing distribution. Employing a stratified split guarantees that the 95 unique website classes ($C$) remain perfectly balanced across all three subsets, ensuring our validation and test evaluations are mathematically un-leaked and unbiased.

We initialize and train a Linear Support Vector Classifier (LinearSVC) using a multi-class One-vs-Rest (OvR) optimization strategy. The model fits a linear hyperplane boundary to separate traffic patterns using our 3 basic features across the $63,438$ **training samples**. Following optimization, we generate a subset report to analyze initial boundaries and validation metrics.

> 📌 **Note:** The two code blocks immediately following this section are included purely to ensure reproducibility. They reiterate the initial raw data loading, sign-extraction, and fixed-length sequence standardization steps established during Phase 1. If your current kernel session already contains the preprocessed global variables `X_clean` and `y_raw` in memory, these initial lines may be skipped.

In [1]:
!unzip data/CW.npz.zip -d data/

import numpy as np

# Load the archive using the exact path revealed by your unzip command
data_path = 'data/datasets/Defense/CW.npz'
data_archive = np.load(data_path, allow_pickle=True)

# Extract the feature matrix X and target labels y
X_raw = data_archive['X']
y_raw = data_archive['y']

# Print out the dataset statistics for your progress report
print("\n\nDataset Statistics: ")
print(f"Total Traffic Traces (Rows): {X_raw.shape[0]}")
print(f"Target Labels Shape: {y_raw.shape[0]}")
print(f"Number of unique websites (classes): {len(np.unique(y_raw))}")

# Inspect a single raw sample trace profile
print("\n\nRaw Sample Inspection: ")
print(f"Shape of the very first raw trace: {X_raw[0].shape}")
print(f"First 10 elements of sample 0: {X_raw[0][:10]}")

Archive:  data/CW.npz.zip
  inflating: data/datasets/Defense/CW.npz  


Dataset Statistics: 
Total Traffic Traces (Rows): 105730
Target Labels Shape: 105730
Number of unique websites (classes): 95


Raw Sample Inspection: 
Shape of the very first raw trace: (10000,)
First 10 elements of sample 0: [ 1.00000000e-06 -9.28558873e-01  9.99152945e-01 -1.03146009e+00
  3.76949601e+00 -3.89432101e+00  3.89727907e+00 -4.02370195e+00
  4.03020387e+00 -4.15452604e+00]


In [2]:
import numpy as np

print("Standardizing sequences to length 5000 and extracting direction vectors...")

# Define the target length
MAX_LEN = 5000

# Initialize clean matrix with float32 to conserve Colab RAM memory
X_clean = np.zeros((X_raw.shape[0], MAX_LEN), dtype=np.float32)

# Loop through and standardize each traffic trace profile
for i in range(X_raw.shape[0]):
    directions = np.sign(X_raw[i])

    length_to_copy = min(len(directions), MAX_LEN)

    # Insert the trace into our clean matrix (any remaining space stays 0 padding)
    X_clean[i, :length_to_copy] = directions[:length_to_copy]

print("\nPreprocessing Complete: ")
print(f"Final Cleaned Feature Matrix Shape: {X_clean.shape}")
print(f"First 10 elements of cleaned sample 0: {X_clean[0][:10]}")

Standardizing sequences to length 5000 and extracting direction vectors...

Preprocessing Complete: 
Final Cleaned Feature Matrix Shape: (105730, 5000)
First 10 elements of cleaned sample 0: [ 1. -1.  1. -1.  1. -1.  1. -1.  1. -1.]


> 📌 **Note:** Below is the implementation for the baseline SVM Classification pipeline. This phase partitions features into traning, validation, and testing to guarantee stable website class distribution. Finally, it outputs the inital **benchmark classification accuracy**.

In [5]:
from sklearn.model_selection import train_test_split
import numpy as np

print("Extracting 3 statistical features for the SVM baseline...")

# Compute aggregate statistical features using optimized column-wise operations (axis=1)
outgoing_counts = np.sum(X_clean == 1.0, axis=1)
incoming_counts = np.sum(X_clean == -1.0, axis=1)
total_volumes = outgoing_counts + incoming_counts

# Combine the features horizontally into an (N, 3) matrix
X_svm_features = np.column_stack((total_volumes, outgoing_counts, incoming_counts))

print(f"SVM Feature Matrix Shape: {X_svm_features.shape}")
print(f"Sample 0 Feature Check [Volume, Out, In]: {X_svm_features[0]}")  # [2861  465  2396]

print("\nExecuting Stratified Dataset Splits (60/20/20)...")

# Step 1: Hold out 20% of the data for the final absolute evaluation Test set
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X_svm_features, y_raw, test_size=0.20, random_state=42, stratify=y_raw
)

# Step 2: Split the remaining 80% into Train (75%) and Validation (25%)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)

print(f"Training Set Size:   {X_train.shape[0]} samples") # 63438 samples
print(f"Validation Set Size: {X_val.shape[0]} samples")   # 21146 samples
print(f"Testing Set Size:    {X_test.shape[0]} samples")  # 21146 samples

Extracting 3 statistical features for the SVM baseline...
SVM Feature Matrix Shape: (105730, 3)
Sample 0 Feature Check [Volume, Out, In]: [2861  465 2396]

Executing Stratified Dataset Splits (60/20/20)...
Training Set Size:   63438 samples
Validation Set Size: 21146 samples
Testing Set Size:    21146 samples


In [6]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report
import time

print("Initializing Linear Support Vector Classifier (Baseline Model)...")
# We set dual=False because our number of samples (63438) is greater than our features (3)
baseline_svm = LinearSVC(dual=False, random_state=42, max_iter=2000)

print("Training baseline model on 63,438 samples.")
start_time = time.time()
baseline_svm.fit(X_train, y_train)
end_time = time.time()

print(f"Training completed in {end_time - start_time:.2f} seconds!")

print("\nEvaluating baseline model on the Validation Set.")
y_val_pred = baseline_svm.predict(X_val)

# Calculate performance metrics
baseline_accuracy = accuracy_score(y_val, y_val_pred)
print(f"Baseline Validation Accuracy: {baseline_accuracy * 100:.2f}%")

# Print a snapshot of the classification report for the progress report
print("\nClassification Report Snippet (First 5 Classes):")
print(classification_report(y_val, y_val_pred,
                            target_names=[f"Website {i}" for i in range(95)],
                            labels=list(range(5))))

# Finally, the Baseline Validation Accuracy is 9.72%.

Initializing Linear Support Vector Classifier (Baseline Model)...
Training baseline model on 63,438 samples.
Training completed in 20.51 seconds!

Evaluating baseline model on the Validation Set.
Baseline Validation Accuracy: 9.72%

Classification Report Snippet (First 5 Classes):
              precision    recall  f1-score   support

   Website 0       0.00      0.00      0.00       223
   Website 1       0.00      0.00      0.00       225
   Website 2       0.12      0.02      0.03       225
   Website 3       0.11      0.00      0.01       225
   Website 4       0.00      0.00      0.00       225

   micro avg       0.07      0.00      0.01      1123
   macro avg       0.05      0.00      0.01      1123
weighted avg       0.05      0.00      0.01      1123



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2687: UserWarning: labels size, 5, does not match size of target_names, 95
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_d